# Training NOCD on Microsoft Academic Graph

This notebook demonstrates training the Neural Overlapping Community Detection
model on the `mag_cs` (Computer Science) dataset, reproducing the original
interactive notebook with the modernized codebase.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from nocd import NOCD
from nocd.data import load_dataset
from nocd.metrics import overlapping_nmi, symmetric_jaccard, evaluate_unsupervised
from nocd.utils import plot_sparse_clustered_adjacency

## Load the dataset

- `A` (adjacency matrix): `scipy.sparse.csr_matrix` of size `[N, N]`
- `X` (attribute matrix): `scipy.sparse.csr_matrix` of size `[N, D]`
- `Z_gt` (ground truth communities): `np.ndarray` of size `[N, K]`

In [ ]:
graph = load_dataset('data/mag_cs.npz')
A, X, Z_gt = graph['A'], graph['X'], graph['Z']
N, K = Z_gt.shape
print(f'Nodes: {N}, Features: {X.shape[1]}, Communities: {K}, Edges: {A.nnz}')

## Train the model

Using the scikit-learn compatible `NOCD` estimator with the same
hyperparameters as the original notebook (GCN + batch norm).

In [ ]:
model = NOCD(
    num_communities=K,
    model_type='gcn',
    hidden_dims=(128,),
    batch_norm=True,
    dropout=0.5,
    lr=1e-3,
    weight_decay=1e-2,
    max_epochs=500,
    balance_loss=True,
    stochastic_loss=True,
    batch_size=20000,
)
model.fit(A, X, y=Z_gt)

## Evaluate

In [ ]:
Z_pred = model.predict(A, X)
Z_soft = model.predict_proba(A, X)

nmi = overlapping_nmi(Z_pred, Z_gt)
jac = symmetric_jaccard(Z_pred, Z_gt)
unsup = evaluate_unsupervised(Z_pred, A)

print(f'Overlapping NMI: {nmi:.4f}')
print(f'Symmetric Jaccard: {jac:.4f}')
for k, v in unsup.items():
    print(f'{k}: {v:.4f}')

## Visualize the adjacency matrix sorted by communities

Left: predicted communities. Right: ground truth.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(20, 10))

# Predicted
z_pred = np.argmax(Z_soft, axis=1)
o_pred = np.argsort(z_pred)
plot_sparse_clustered_adjacency(A, K, z_pred, o_pred, ax=axes[0], markersize=0.05)
axes[0].set_title('Predicted Communities', fontsize=14)

# Ground truth
z_gt = np.argmax(Z_gt, axis=1)
o_gt = np.argsort(z_gt)
plot_sparse_clustered_adjacency(A, K, z_gt, o_gt, ax=axes[1], markersize=0.05)
axes[1].set_title('Ground Truth Communities', fontsize=14)

plt.tight_layout()
plt.show()

## Save and reload

The model can be saved and loaded for later use.

In [ ]:
model.save('/tmp/nocd_demo.pt')

loaded = NOCD.load('/tmp/nocd_demo.pt')
Z_loaded = loaded.predict_proba(A, X)
print('Predictions match:', np.allclose(Z_soft, Z_loaded, atol=1e-5))